## `Материалы кафедры ММП факультета ВМК МГУ. Введение в эффективные системы глубокого обучения.`

## `Задание 04.1. Программирование на Triton`.

#### Фамилия, имя:

Дата выдачи: <span style="color:red">__20 октября__</span>.

Мягкий дедлайн: <span style="color:red">__5 ноября 15:00__</span>.

Стоимость: __1 балл__ (дополнительные задания).

<span style="color:red">__В ноутбуке все клетки должны выполняться без ошибок при последовательном их выполнении.__</span>

#### `Москва, 2026`

##### По всем вопросам писать `@trandelik`

## Введение

В рамках этого задания вам предстоит познакомиться с основными конструкциями Triton, чтобы затем воспользоваться ими при реализации матричного умножения. Сам алгоритм tiled matrix multiplication вы уже писали на CUDA, теперь давайте потренируемся писать на Triton на простых задачах.

Начнем с готового кернела сложения векторов. Затем реализуем транспонирование матрицы, сумму по столбцам и внешнее произведение векторов.

Документация [Triton Python API](https://triton-lang.org/main/python-api/triton.language.html)

**Важно:** авторы настоятельно рекомендуют выполнять домашнее задание в Google Colab и не гарантируют поддержку других сред. Также мы приведем версии CUDA, драйверов, python и библиотек ниже.

## Setup

In [1]:
!nvidia-smi

Tue Oct 20 18:44:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:58:59_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0


 Мы тестировали код на `Python 3.13.15` и следующими версиями библиотек:

```
- torch==2.6.0
- torchaudio==2.6.0
- torchvision==0.21.0
- transformers==4.56.1
- datasets==3.6.0
- tqdm==4.67.1
- comet_ml==3.51.0
- seqeval==1.2.2
- accelerate-1.10.1
- triton==2.3.0
```

In [ ]:
!python --version

In [ ]:
%pip freeze | grep datasets==
%pip freeze | grep triton==
%pip freeze | grep torch==
%pip freeze | grep seqeval==
%pip freeze | grep transformers==
%pip freeze | grep accelerate==

In [ ]:
# Этот код может вам помочь установить нужные библиотеки. Лучше использовать Google Colab.

# Можете использовать --index-url https://download.pytorch.org/whl/cuXX если считаете нужным
# Вместо XX подставить версию драйвера из nvidia-smi
%pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 -q
%pip install triton==3.2.0 -q
%pip install datasets==3.6.0  transformers==4.56.1 huggingface-hub==0.35.1 accelerate==1.10.1 -q
%pip install "setuptools<74" wheel
%pip install seqeval==1.2.2 --no-build-isolation

In [ ]:
%pip freeze | grep datasets==
%pip freeze | grep triton==
%pip freeze | grep torch==
%pip freeze | grep seqeval==
%pip freeze | grep transformers==
%pip freeze | grep accelerate==

In [4]:
import sys

import matplotlib.pyplot as plt
import torch
import triton
import triton.testing
import triton.language as tl

assert torch.cuda.is_available(), (
    "GPU недоступен."
)

DEVICE = torch.device("cuda")
torch.manual_seed(42)

print("Python version:", sys.version.split()[0])
print("Torch version:", torch.__version__)
print("Triton version:", triton.__version__)
print("CUDA version:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(DEVICE))

Python version: 3.13.15
Torch version: 2.6.0+cu124
Triton version: 3.2.0
CUDA version: 12.4
GPU: Tesla T4


### Как писать кернелы?

В CUDA нужно было описывать работу отдельного потока. В Triton же нужно описывать работу **одной программы**, которая обрабатывает сразу блок элементов. Распределением этих элементов по потокам займется компилятор.

Номер программы можно получить через `tl.program_id`, а индексы элементов внутри блока &ndash; через `tl.arange`. Например, `tl.arange(0, BLOCK_SIZE)` создает вектор индексов от 0 до `BLOCK_SIZE - 1`. Размер блока передается как `tl.constexpr`, чтобы он был известен при компиляции.

Для чтения и записи можно использовать `tl.load` и `tl.store`. В обе функции можно передать маску: она указывает, по каким адресам разрешено обращаться к памяти. У `tl.load` есть еще аргумент `other` &ndash; значение для элементов, которые не были прочитаны из-за маски.

**Важно:** в кернелах размеры блоков для `tl.arange` должны быть степенями двойки. Размеры самих векторов и матриц могут быть любыми положительными числами.

Код запуска кернелов и тесты мы подготовили за вас. Заполните места с `### YOUR CODE HERE`. После реализации кернела удалите `raise NotImplementedError(...)` из соответствующей обертки и запустите тесты. До этого тесты будут сообщать, что решение еще не написано.

## Задание 1. Understanding the Kernel (0.2 балла)

Первая задача &ndash; разобраться, как работает простой кернел на Triton. Для этого рассмотрим сложение двух векторов:

$$z_i = x_i + y_i.$$

Каждая программа будет вычислять `BLOCK_SIZE` элементов результата.

Кернел ниже уже написан. Ваша задача &ndash; ответить на вопросы о реализации.

In [ ]:
@triton.jit
def vector_add_kernel(x_ptr, y_ptr, z_ptr, N, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(axis=0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < N

    x = tl.load(x_ptr + offsets, mask=mask, other=0.0)
    y = tl.load(y_ptr + offsets, mask=mask, other=0.0)
    z = x + y

    tl.store(z_ptr + offsets, z, mask=mask)


def vector_add(x, y, block_size=4):
    assert x.is_cuda and y.is_cuda and x.device == y.device
    assert x.ndim == y.ndim == 1 and x.shape == y.shape
    assert x.is_contiguous() and y.is_contiguous()
    assert x.dtype == y.dtype == torch.float32 and x.numel() > 0
    assert block_size > 0 and (block_size & (block_size - 1)) == 0

    z = torch.empty_like(x)
    grid = (triton.cdiv(x.numel(), block_size),)
    vector_add_kernel[grid](x, y, z, x.numel(), BLOCK_SIZE=block_size)
    return z

Пусть длина векторов `N = 10`, а размер блока `BLOCK_SIZE = 4`. Ответьте на следующие вопросы:

1. Сколько программ будет запущено? Выпишите `offsets` и `mask` для каждой программы в таблице ниже.
2. Какая программа вычислит `z[7]`? Какой это элемент внутри ее блока, если считать с нуля?
3. Что изменится при `BLOCK_SIZE = 8`? А при `BLOCK_SIZE = 16`? Для каждого случая укажите количество программ и маску последней программы.
4. Зачем передавать маску и в `tl.load`, и в `tl.store`? Можно ли убрать маску из загрузки, если оставить `other=0`?
5. Почему размер grid вычисляется через `triton.cdiv`, а не через `N // BLOCK_SIZE`?
6. Означает ли `BLOCK_SIZE = 4`, что программа использует ровно четыре CUDA-потока?

**Ваши ответы**:

| `pid` | `offsets` | `mask` |
| --- | --- | --- |
| 0 | ... | ... |
| 1 | ... | ... |
| 2 | ... | ... |

1. ...
2. ...
3. ...
4. ...
5. ...
6. ...

## Задание 2. Транспонирование матрицы (0.4 балла)

Теперь напишем свой кернел для двумерных данных.

Наша задача &ndash; транспонировать матрицу `X` размера `[M, N]`. Результатом будет матрица `Y` размера `[N, M]`:

$$Y_{j,i} = X_{i,j}.$$

Для этого мы будем использовать **двумерный grid программ**. Каждой программе поручим загрузить один тайл входной матрицы и записать его элементы в соответствующие позиции выходной.

**Важно:** нужно выделить новую память и записать в нее результат.

### Как читать двумерный тайл

Так как в Triton мы работаем с указателями,нам нужно уметь вычислять адрес элемента в памяти для матриц. Для этого воспользуемся **strides** (следующая формула верна только для contiguous тензоров):

```python
x_ptr + i * stride_xm + j * stride_xn
```

`stride_xm` показывает, на сколько элементов нужно сдвинуть адрес при переходе к следующей строке, а `stride_xn` &ndash; к следующему столбцу. У непрерывной матрицы `[M, N]` они равны `N` и `1`. Strides измеряются в элементах, а не в байтах.

Теперь построим сразу блок адресов. Для этого понадобятся **broadcasting** и добавление осей:

1. Вектор `rows` имеет форму `[BLOCK_M]`. После `rows[:, None]` получится `[BLOCK_M, 1]`.
2. Вектор `cols` имеет форму `[BLOCK_N]`. После `cols[None, :]` получится `[1, BLOCK_N]`.
3. При сложении выражений с этими формами получится блок `[BLOCK_M, BLOCK_N]`.

Таким образом можно получить адреса всех элементов тайла и выполнить одну двумерную загрузку через `tl.load`.

### Testing Utils

Мы подготовили тестовую функцию для проверки транспонирования. Сначала проверим маленькую матрицу, затем &ndash; случайные матрицы разных размеров. В тестах будут прямоугольные матрицы и размеры, которые не делятся на размер тайла.

Основные тесты используют непрерывные входы. Дополнительную проверку входов с произвольными strides можно включить через `check_strides=True`.

In [ ]:
def test_transpose(custom_function, check_strides=False):
    print("--- Testing: Transpose ---")

    def check(x, block_m=16, block_n=32):
        before = x.clone()
        result = custom_function(x, block_m=block_m, block_n=block_n)
        assert result.shape == (x.shape[1], x.shape[0])
        assert result.dtype == x.dtype and result.device == x.device
        assert result.is_contiguous(), "Выход должен быть contiguous"
        assert result.untyped_storage().data_ptr() != x.untyped_storage().data_ptr(), (
            "Нужно выделить новую память, а не вернуть представление входа"
        )
        torch.testing.assert_close(result, x.T.contiguous(), rtol=0, atol=0)
        torch.testing.assert_close(x, before, rtol=0, atol=0)

    example = torch.tensor([[1, 2, 3], [4, 5, 6]], device=DEVICE, dtype=torch.float32)
    check(example)

    for dtype in [torch.float32, torch.float16]:
        for M, N in [(1, 1), (1, 37), (29, 1), (16, 32), (31, 47), (65, 97), (128, 256)]:
            x = torch.randn((M, N), device=DEVICE, dtype=dtype)
            check(x)
            check(x, block_m=32, block_n=16)

    if check_strides:
        base = torch.randn((67, 103), device=DEVICE)
        for x in [base.T, base[1::2, 2::3]]:
            check(x)

    print("All tests passed!")

### Reference Solution

In [ ]:
def torch_transpose(x, block_m=16, block_n=32):
    # Параметры блоков нужны только для общего интерфейса с тестами.
    # clone гарантирует новую память и для матриц с одной строкой/столбцом.
    return x.T.clone(memory_format=torch.contiguous_format)


test_transpose(torch_transpose)

### Задание 2.1. Transpose with Tiling (0.2 балла)

В первом домашнем задании вы уже использовали тайлы при умножении матриц. Давайте воспользуемся тем же разбиением для транспонирования.

**Алгоритм:**

1. Получаем номер программы по каждой из двух осей grid.
2. Вычисляем индексы строк и столбцов входного тайла.
3. Строим двумерный блок адресов и загружаем данные. Не забываем про маску по обеим осям.
4. Вычисляем адреса в выходной матрице и записываем результат.
**Подсказка:** можно сохранить форму загруженного тайла и изменить только адреса записи.

In [ ]:
@triton.jit
def transpose_kernel(
    x_ptr, y_ptr,
    M, N,
    stride_xm, stride_xn,
    stride_yn, stride_ym,
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr,
):
    # 1. Номера программы и индексы строк и столбцов входного тайла.
    ### YOUR CODE HERE

    # 2. Двумерные адреса, маска и загрузка тайла.
    ### YOUR CODE HERE

    # 3. Адреса в выходной матрице и запись результата с маской.
    ### YOUR CODE HERE
    pass



def transpose_triton(x, block_m=16, block_n=32):
    assert x.is_cuda and x.ndim == 2 and x.numel() > 0
    assert x.dtype in (torch.float32, torch.float16)

    raise NotImplementedError('Заполните transpose_kernel и удалите эту строку')

    M, N = x.shape
    y = torch.empty((N, M), device=x.device, dtype=x.dtype)
    grid = (triton.cdiv(M, block_m), triton.cdiv(N, block_n))
    transpose_kernel[grid](
        x, y, M, N,
        x.stride(0), x.stride(1),
        y.stride(0), y.stride(1),
        BLOCK_M=block_m, BLOCK_N=block_n,
    )
    return y

In [ ]:
test_transpose(transpose_triton)

x = torch.tensor([[1, 2, 3], [4, 5, 6]], device=DEVICE, dtype=torch.float32)
print("X:\n", x.cpu())
print("Y:\n", transpose_triton(x).cpu())

### Автоматический подбор параметров

Наш кернел уже работает. Но какой размер тайла лучше выбрать? А сколько warp-ов должна использовать одна программа? Перебирать эти параметры вручную неудобно, особенно если размеры матриц меняются.

Для этого в Triton есть [`triton.autotune`](https://triton-lang.org/main/python-api/generated/triton.autotune.html). Передадим ему несколько конфигураций, а он измерит время работы каждой и выберет самую быструю для текущей конфигурации входа (ключ).

Подробнее о параметрах: [`triton.Config`](https://triton-lang.org/main/python-api/generated/triton.Config.html).

В ключ включим размеры матрицы и strides входа. У обычной матрицы и ее транспонированного представления разный порядок доступа к памяти, даже если размеры совпадают.

Код подбора параметров мы подготовили за вас. `transpose_autotuned_kernel` вызывает уже написанный `transpose_kernel`, поэтому формулы адресов повторять не нужно. Обратите внимание на `grid`: теперь он зависит от `META`, потому что размеры тайла выбирает autotune.

**Важно:** первый запуск будет дольше из-за компиляции и перебора конфигураций. Во время подбора кернел выполняется несколько раз.

In [ ]:
@triton.autotune(
    configs=[
        triton.Config({"BLOCK_M": 16, "BLOCK_N": 32}, num_warps=4),
        triton.Config({"BLOCK_M": 32, "BLOCK_N": 16}, num_warps=4),
        triton.Config({"BLOCK_M": 32, "BLOCK_N": 32}, num_warps=4),
        triton.Config({"BLOCK_M": 32, "BLOCK_N": 64}, num_warps=8),
    ],
    key=["M", "N", "stride_xm", "stride_xn"],
)
@triton.jit
def transpose_autotuned_kernel(
    x_ptr, y_ptr,
    M, N,
    stride_xm, stride_xn,
    stride_yn, stride_ym,
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr,
):
    # Используем тот же кернел с параметрами, выбранными autotune.
    transpose_kernel(
        x_ptr, y_ptr, M, N,
        stride_xm, stride_xn, stride_yn, stride_ym,
        BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N,
    )


def transpose_triton_autotuned(x):
    assert x.is_cuda and x.ndim == 2 and x.numel() > 0
    assert x.dtype in (torch.float32, torch.float16)

    M, N = x.shape
    y = torch.empty((N, M), device=x.device, dtype=x.dtype)
    grid = lambda META: (
        triton.cdiv(M, META["BLOCK_M"]),
        triton.cdiv(N, META["BLOCK_N"]),
    )
    transpose_autotuned_kernel[grid](
        x, y, M, N,
        x.stride(0), x.stride(1),
        y.stride(0), y.stride(1),
    )
    return y


def test_transpose_autotuned(custom_function, check_strides=False):
    print("--- Testing: Autotuned Transpose ---")
    for dtype in [torch.float32, torch.float16]:
        base = torch.randn((31, 47), device=DEVICE, dtype=dtype)
        inputs = [torch.ones((1, 1), device=DEVICE, dtype=dtype), base]
        if check_strides:
            inputs.extend([base.T, base[1::2, 2::3]])
        for x in inputs:
            before = x.clone()
            result = custom_function(x)
            assert result.shape == (x.shape[1], x.shape[0])
            assert result.dtype == x.dtype and result.device == x.device
            assert result.is_contiguous()
            assert result.untyped_storage().data_ptr() != x.untyped_storage().data_ptr()
            torch.testing.assert_close(result, x.T.contiguous(), rtol=0, atol=0)
            torch.testing.assert_close(x, before, rtol=0, atol=0)
    print("All tests passed!")


test_transpose_autotuned(transpose_triton_autotuned)

### Сравнение результатов

Теперь сравним время работы трех реализаций: PyTorch, нашего кернела с фиксированным тайлом $16\times32$ и того же кернела с autotune. Для этого воспользуемся `triton.testing.do_bench`, который выполняет прогрев и измеряет время на GPU.

Перед замерами отдельно вызовем каждую реализацию. Так компиляция и подбор конфигурации завершатся до измерения. Все входные данные уже находятся на GPU, переносы между CPU и GPU в замеры не входят. Время вызова включает работу с результатом так, как она написана в соответствующей обертке.

In [ ]:
transpose_results = []


def benchmark_transpose(sizes=(256, 512, 1024)):
    transpose_results.clear()
    implementations = {
        "PyTorch": torch_transpose,
        "Triton, fixed tile": transpose_triton,
        "Triton, autotune": transpose_triton_autotuned,
    }

    for size in sizes:
        x = torch.randn((size, size), device=DEVICE)
        for name, custom_function in implementations.items():
            custom_function(x)  # Компиляция и autotune до замера.
            ms = triton.testing.do_bench(
                lambda: custom_function(x), warmup=10, rep=30,
            )
            transpose_results.append({"name": name, "size": size, "ms": ms})
            print(f"{name}, {size}x{size}: {ms:.4f} ms")
        print("Best config:", transpose_autotuned_kernel.best_config)

    plt.figure(figsize=(8, 4))
    for name in implementations:
        rows = [row for row in transpose_results if row["name"] == name]
        plt.plot([row["size"] for row in rows], [row["ms"] for row in rows], marker="o", label=name)
    plt.xlabel("Matrix size")
    plt.ylabel("Time, ms")
    plt.title("Matrix Transpose")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()


benchmark_transpose()

### Задание 2.2. Ответьте на следующие вопросы (0.2 балла)

1. Почему для матрицы `[31, 47]` нужна маска по обеим осям?
2. Почему в варианте с autotune grid зависит от `META`? Что произойдет, если оставить grid от фиксированного тайла, а размер тайла будет меняться?
3. Зачем мы включили strides в ключ autotune? Почему только размеров матрицы может быть недостаточно?
4. Почему в нашем случае можно несколько раз запускать кернел при подборе параметров без обнуления результата? Что изменится, если кернел будет прибавлять значения к выходу?

**Ваши ответы**:

1. ...
2. ...
3. ...
4. ...

## Задание 3. Суммы по столбцам (0.2 балла)

В предыдущей части мы загружали тайл и записывали столько же элементов в другую матрицу. Теперь давайте посчитаем сумму элементов каждого столбца:

$$y_j = \sum_{i=0}^{M-1} X_{i,j}.$$

Для входной матрицы `[M, N]` получим вектор длины `N`. Здесь нам понадобится **редукция**, с которой вы уже встречались.

Каждой программе поручим обработать несколько столбцов целиком. Чтобы весь нужный блок можно было загрузить сразу, ограничимся матрицами с **`1 ≤ M ≤ 256`**.

### Testing Utils

Мы подготовили тесты для разных размеров матриц, включая одну строку и один столбец. Суммировать будем в FP32 и возвращать результат в FP32, даже если вход хранится в FP16.

In [ ]:
def test_column_sum(custom_function):
    print("--- Testing: Column Sum ---")

    def check(x, block_n=32):
        before = x.clone()
        result = custom_function(x, block_n=block_n)
        assert result.shape == (x.shape[1],)
        assert result.dtype == torch.float32 and result.device == x.device
        torch.testing.assert_close(result, x.float().sum(dim=0), rtol=1e-5, atol=1e-4)
        torch.testing.assert_close(x, before, rtol=0, atol=0)

    example = torch.tensor([[1, 2, 3], [4, 5, 6]], device=DEVICE, dtype=torch.float32)
    check(example)  # Ожидаем [5, 7, 9].

    for dtype in [torch.float32, torch.float16]:
        for M, N in [(1, 1), (1, 37), (29, 1), (16, 32), (37, 53), (255, 97), (256, 128)]:
            x = torch.randn((M, N), device=DEVICE, dtype=dtype)
            for block_n in [16, 32]:
                check(x, block_n=block_n)
        check(torch.zeros((37, 53), device=DEVICE, dtype=dtype))
        check(-torch.ones((37, 53), device=DEVICE, dtype=dtype))

    # Сумма каждого столбца равна 256000 и не помещается в FP16.
    check(torch.full((256, 35), 1000.0, device=DEVICE, dtype=torch.float16))
    print("All tests passed!")

### Baselines

In [ ]:
def torch_column_sum(x, block_n=32):
    return x.sum(dim=0, dtype=torch.float32)


test_column_sum(torch_column_sum)

В CUDA для редукции приходилось явно организовывать работу потоков и синхронизации. В Triton можно воспользоваться готовой функцией [`tl.sum`](https://triton-lang.org/main/python-api/generated/triton.language.sum.html). Остается правильно загрузить блок и выбрать ось суммирования.

**Алгоритм:**

1. Определяем, за какие `BLOCK_N` столбцов отвечает программа.
2. Строим блок адресов `[BLOCK_M, BLOCK_N]`, где `BLOCK_M` &ndash; число строк, округленное вверх до степени двойки.
3. Загружаем значения с маской по строкам и столбцам. Для лишних элементов выбираем такое значение `other`, которое не изменит сумму.
4. Приводим загруженные значения к `tl.float32` и выполняем редукцию по нужной оси.
5. Записываем `BLOCK_N` сумм с маской по столбцам.

In [ ]:
@triton.jit
def column_sum_kernel(
    x_ptr, y_ptr,
    M, N,
    stride_xm, stride_xn,
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr,
):
    # 1. Индексы всех строк и столбцов текущей программы.
    ### YOUR CODE HERE

    # 2. Адреса, маска и загрузка двумерного блока.
    ### YOUR CODE HERE

    # 3. Приведение к FP32 и редукция по нужной оси.
    ### YOUR CODE HERE

    # 4. Запись вектора сумм с маской по столбцам.
    ### YOUR CODE HERE
    pass



def column_sum_triton(x, block_n=32):
    assert x.is_cuda and x.ndim == 2 and x.numel() > 0
    assert x.dtype in (torch.float32, torch.float16)
    assert 1 <= x.shape[0] <= 256, "В этом упражнении M не больше 256"

    raise NotImplementedError('Заполните column_sum_kernel и удалите эту строку')

    M, N = x.shape
    y = torch.empty((N,), device=x.device, dtype=torch.float32)
    block_m = triton.next_power_of_2(M)
    grid = (triton.cdiv(N, block_n),)
    column_sum_kernel[grid](
        x, y, M, N, x.stride(0), x.stride(1),
        BLOCK_M=block_m, BLOCK_N=block_n,
    )
    return y

In [ ]:
test_column_sum(column_sum_triton)

## Задание 4. Внешнее произведение и масштабирование (0.2 балла)

В предыдущей части мы использовали редукцию, чтобы превратить двумерный блок в вектор. Теперь давайте из двух векторов получим матрицу, а затем воспользуемся этим приемом для масштабирования данных.

Для векторов u длины M и v длины N внешнее произведение определяется так:

$Y_{i,j} = u_i \cdot v_j$.

Например, для $u = [1, 2]$ и $v = [10, 20, 30]$ получим:

$$
Y =
\begin{pmatrix}
10 & 20 & 30 \\
20 & 40 & 60
\end{pmatrix}.
$$

В отличие от скалярного произведения, здесь нет суммирования. Мы перемножаем каждую возможную пару элементов и получаем матрицу размера `[M, N]`. Для этого удобно воспользоваться broadcasting:

```python
y = u[:, None] * v[None, :]
```

Теперь пусть у нас уже есть матрица X размера `[M, N]`. Наша следующая задача &ndash; умножить каждую ее строку на свой коэффициент из u, а каждый столбец &ndash; на свой коэффициент из v:

$Z_{i,j} = X_{i,j} \cdot u_i \cdot v_j.$

Такая операция называется масштабированием матрицы по строкам и столбцам. Например, если u[i] = 2, все элементы i-й строки умножатся на 2. Если при этом v[j] = 3, элемент на пересечении этой строки и j-го столбца умножится на 6.

На PyTorch это можно записать так:

```python
z = (x * u[:, None]) * v[None, :]
```

Можно также сначала вычислить внешнее произведение u и v, а затем поэлементно умножить его на X. Однако тогда придется записать промежуточную матрицу в GPU-память, а потом прочитать ее обратно. Поэтому следует объединить эти действия в одном кернеле: загрузить тайл X, применить коэффициенты и записать только итоговый результат.

### Как это связано с квантизацией?

Рассмотрим симметричную квантизацию, в которой ноль представляется целым нулем. Пусть у нас есть матрицы A размера `[M, K]` и B размера `[K, N]`. Для каждой строки A выберем свой масштаб $s^{A}_i$, а для каждого столбца B &ndash; свой масштаб $s^{B}_j$. Тогда:

$A_{i,k} \approx s^{A}_i \cdot A^{INT8}_{i,k},   B_{k,j} \approx s^{B}_j \cdot B^{INT8}_{k,j}.$

При умножении целочисленных матриц получим аккумулятор:

$S_{i,j} = \sum_k A^{INT8}_{i,k} B^{INT8}_{k,j}.$

Числа в S пока выражены в единицах квантизованных входов. Чтобы получить приближение результата исходного вещественного умножения, нужно применить масштабы:

$(AB)_{i,j} \approx s^{A}_i \cdot s^{B}_j \cdot S_{i,j}.$

Масштаб строки A и масштаб столбца B не зависят от индекса суммирования k, поэтому мы вынесли их за сумму. Получилась ровно та операция, которую мы собираемся реализовать: масштабирование матрицы по строкам и столбцам.

Для линейного слоя A будет матрицей активаций, а B &ndash; транспонированной матрицей весов. Тогда масштабы строк соответствуют отдельным токенам, а масштабы столбцов &ndash; выходным каналам слоя.

Сначала реализуем внешнее произведение. Затем расширим кернел: добавим загрузку матрицы и применим к ней масштабы. Если вход хранится в INT32, перед умножением на вещественные коэффициенты приведем его элементы к FP32.

### Testing Utils

Векторы `u` и `v` хранятся в FP32. Матрица `X` во второй части может храниться в FP32, FP16 или INT32. Перед умножением ее элементы нужно привести к FP32. Результат в обеих частях тоже возвращаем в FP32.

In [ ]:
def test_outer_product(custom_function):
    print("--- Testing: Outer Product ---")

    def check(u, v, block_m=16, block_n=32):
        u_before, v_before = u.clone(), v.clone()
        result = custom_function(u, v, block_m=block_m, block_n=block_n)
        assert result.shape == (u.numel(), v.numel())
        assert result.dtype == torch.float32 and result.device == u.device
        torch.testing.assert_close(result, u[:, None] * v[None, :], rtol=1e-5, atol=1e-6)
        torch.testing.assert_close(u, u_before, rtol=0, atol=0)
        torch.testing.assert_close(v, v_before, rtol=0, atol=0)

    check(
        torch.tensor([1.0, 2.0], device=DEVICE),
        torch.tensor([10.0, 20.0, 30.0], device=DEVICE),
    )
    for M, N in [(1, 1), (1, 37), (29, 1), (16, 32), (31, 47), (65, 97)]:
        u = torch.randn(M, device=DEVICE)
        v = torch.randn(N, device=DEVICE)
        check(u, v)
        check(u, v, block_m=32, block_n=16)
    check(torch.zeros(19, device=DEVICE), torch.randn(37, device=DEVICE))
    check(-torch.ones(19, device=DEVICE), torch.ones(37, device=DEVICE))
    print("All tests passed!")


def test_scale_matrix(custom_function):
    print("--- Testing: Scale Matrix ---")

    def check(x, u, v, block_m=16, block_n=32):
        x_before, u_before, v_before = x.clone(), u.clone(), v.clone()
        result = custom_function(x, u, v, block_m=block_m, block_n=block_n)
        assert result.shape == x.shape
        assert result.dtype == torch.float32 and result.device == x.device
        assert result.untyped_storage().data_ptr() != x.untyped_storage().data_ptr()
        expected = (x.float() * u[:, None]) * v[None, :]
        torch.testing.assert_close(result, expected, rtol=1e-5, atol=1e-5)
        torch.testing.assert_close(x, x_before, rtol=0, atol=0)
        torch.testing.assert_close(u, u_before, rtol=0, atol=0)
        torch.testing.assert_close(v, v_before, rtol=0, atol=0)

    for dtype in [torch.float32, torch.float16, torch.int32]:
        for M, N in [(1, 1), (1, 37), (29, 1), (16, 32), (31, 47), (65, 97)]:
            if dtype == torch.int32:
                x = torch.randint(-1000, 1001, (M, N), device=DEVICE, dtype=dtype)
            else:
                x = torch.randn((M, N), device=DEVICE, dtype=dtype)
            u = torch.randn(M, device=DEVICE)
            v = torch.randn(N, device=DEVICE)
            check(x, u, v)
            check(x, u, v, block_m=32, block_n=16)
            check(x, torch.ones_like(u), torch.ones_like(v))
            check(x, torch.zeros_like(u), v)
            check(x, u, torch.zeros_like(v))

    print("All tests passed!")

### References

In [ ]:
def torch_outer_product(u, v, block_m=16, block_n=32):
    return u[:, None] * v[None, :]


def torch_scale_matrix(x, u, v, block_m=16, block_n=32):
    return (x.float() * u[:, None]) * v[None, :]


test_outer_product(torch_outer_product)
test_scale_matrix(torch_scale_matrix)

Реализуем внешнее произведение векторов. Как и при транспонировании, каждой программе поручим вычислить один тайл результата.

**Алгоритм:**

1. Вычисляем индексы строк и столбцов выходного тайла.
2. Загружаем нужные элементы `u` и `v`. Для каждого вектора используем свою одномерную маску.
3. Добавляем оси через `[:, None]` и `[None, :]`, затем перемножаем векторы с broadcasting.
4. Записываем тайл результата с двумерной маской.

**Важно:** здесь достаточно поэлементного умножения. `tl.dot` и `tl.sum` не понадобятся.

In [ ]:
@triton.jit
def outer_product_kernel(
    u_ptr, v_ptr, y_ptr,
    M, N,
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr,
):
    # 1. Индексы строк и столбцов выходного тайла.
    ### YOUR CODE HERE

    # 2. Загрузка частей двух векторов с одномерными масками.
    ### YOUR CODE HERE

    # 3. Внешнее произведение с помощью broadcasting.
    ### YOUR CODE HERE

    # 4. Двумерные адреса и запись результата с маской.
    ### YOUR CODE HERE
    pass



def outer_product_triton(u, v, block_m=16, block_n=32):
    assert u.is_cuda and v.is_cuda and u.device == v.device
    assert u.ndim == v.ndim == 1 and u.numel() > 0 and v.numel() > 0
    assert u.dtype == v.dtype == torch.float32
    assert u.is_contiguous() and v.is_contiguous()

    raise NotImplementedError('Заполните outer_product_kernel и удалите эту строку')

    M, N = u.numel(), v.numel()
    y = torch.empty((M, N), device=u.device, dtype=torch.float32)
    grid = (triton.cdiv(M, block_m), triton.cdiv(N, block_n))
    outer_product_kernel[grid](
        u, v, y, M, N, BLOCK_M=block_m, BLOCK_N=block_n,
    )
    return y

In [ ]:
test_outer_product(outer_product_triton)

Внешнее произведение готово. Давайте воспользуемся им для масштабирования матрицы по строкам и столбцам.

In [ ]:
@triton.jit
def scale_matrix_kernel(
    x_ptr, u_ptr, v_ptr, z_ptr,
    M, N,
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr,
):
    # 1. Индексы строк и столбцов, маски и загрузка масштабов.
    ### YOUR CODE HERE

    # 2. Загрузка тайла матрицы и приведение значений к FP32.
    ### YOUR CODE HERE

    # 3. Масштабирование по строкам и столбцам с broadcasting.
    ### YOUR CODE HERE

    # 4. Запись результата с двумерной маской.
    ### YOUR CODE HERE
    pass



def scale_matrix_triton(x, u, v, block_m=16, block_n=32):
    assert x.is_cuda and u.is_cuda and v.is_cuda
    assert x.device == u.device == v.device
    assert x.ndim == 2 and u.ndim == v.ndim == 1 and x.numel() > 0
    assert x.shape == (u.numel(), v.numel())
    assert x.dtype in (torch.float32, torch.float16, torch.int32)
    assert u.dtype == v.dtype == torch.float32
    assert x.is_contiguous() and u.is_contiguous() and v.is_contiguous()

    raise NotImplementedError('Заполните scale_matrix_kernel и удалите эту строку')

    M, N = x.shape
    z = torch.empty((M, N), device=x.device, dtype=torch.float32)
    grid = (triton.cdiv(M, block_m), triton.cdiv(N, block_n))
    scale_matrix_kernel[grid](
        x, u, v, z, M, N, BLOCK_M=block_m, BLOCK_N=block_n,
    )
    return z

In [ ]:
test_scale_matrix(scale_matrix_triton)